In [1]:
"""
database.py — MySQL connection and schema for QubitTelemetry
Supports three models: Model A (historical), Model B (7-day), Model C (30-day)
"""

import os
import mysql.connector
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()


# ── Connection ────────────────────────────────────────────────────────────────
def get_connection():
    return mysql.connector.connect(
        host     = os.environ.get("MYSQL_HOST"),
        port     = int(os.environ.get("MYSQL_PORT", 3306)),
        user     = os.environ.get("MYSQL_USER"),
        password = os.environ.get("MYSQL_PASSWORD"),
        database = os.environ.get("MYSQL_DATABASE"),
    )


# ── Schema ────────────────────────────────────────────────────────────────────
CREATE_CALIBRATION = """
CREATE TABLE IF NOT EXISTS calibration_history (
    id             INT AUTO_INCREMENT PRIMARY KEY,
    backend        VARCHAR(50)  NOT NULL,
    qubit          INT          NOT NULL,
    snapshot_date  DATE         NOT NULL,
    backend_ts     DATETIME     NOT NULL,
    T1_us          FLOAT,
    T2_us          FLOAT,
    readout_error  FLOAT,
    collected_at   DATETIME     DEFAULT CURRENT_TIMESTAMP,
    UNIQUE KEY uq_backend_qubit_date (backend, qubit, snapshot_date)
);
"""

# New: one rankings table per model_name
# model_name = 'model_a' | 'model_b' | 'model_c'
CREATE_RANKINGS = """
CREATE TABLE IF NOT EXISTS qubit_rankings (
    id              INT AUTO_INCREMENT PRIMARY KEY,
    model_name      VARCHAR(20)  NOT NULL DEFAULT 'model_a',
    backend         VARCHAR(50)  NOT NULL,
    qubit           INT          NOT NULL,
    snapshot_date   DATE         NOT NULL,
    viability_score FLOAT        NOT NULL,
    viability_rank  INT          NOT NULL,
    label           TINYINT      NOT NULL,
    T1_us           FLOAT,
    T2_us           FLOAT,
    readout_error   FLOAT,
    created_at      DATETIME     DEFAULT CURRENT_TIMESTAMP,
    UNIQUE KEY uq_model_backend_qubit_date (model_name, backend, qubit, snapshot_date)
);
"""

# Migration: add model_name column if upgrading from old schema
MIGRATE_ADD_MODEL_NAME = """
ALTER TABLE qubit_rankings
ADD COLUMN IF NOT EXISTS model_name VARCHAR(20) NOT NULL DEFAULT 'model_a'
AFTER id;
"""

MIGRATE_UPDATE_UNIQUE = """
-- Run manually if old unique key exists without model_name:
-- ALTER TABLE qubit_rankings DROP INDEX uq_backend_qubit_date;
-- ALTER TABLE qubit_rankings ADD UNIQUE KEY uq_model_backend_qubit_date
--   (model_name, backend, qubit, snapshot_date);
"""


def setup_database():
    """Create tables and run migrations. Safe to call on every startup."""
    conn = get_connection()
    cur  = conn.cursor()
    cur.execute(CREATE_CALIBRATION)
    cur.execute(CREATE_RANKINGS)
    # Try migration — fails silently if column already exists
    try:
        cur.execute(MIGRATE_ADD_MODEL_NAME)
    except Exception:
        pass
    conn.commit()
    cur.close()
    conn.close()
    print("Database tables ready.")


# ── Write operations ──────────────────────────────────────────────────────────
def insert_calibration_rows(rows: list) -> int:
    if not rows:
        return 0
    sql = """
        INSERT IGNORE INTO calibration_history
            (backend, qubit, snapshot_date, backend_ts, T1_us, T2_us, readout_error)
        VALUES
            (%(backend)s, %(qubit)s, %(snapshot_date)s, %(backend_ts)s,
             %(T1_us)s, %(T2_us)s, %(readout_error)s)
    """
    conn = get_connection()
    cur  = conn.cursor()
    cur.executemany(sql, rows)
    inserted = cur.rowcount
    conn.commit()
    cur.close()
    conn.close()
    return inserted


def insert_ranking_rows(rows: list, model_name: str = 'model_a') -> int:
    """
    Insert rankings for a specific model.
    model_name: 'model_a' | 'model_b' | 'model_c'
    """
    if not rows:
        return 0
    for row in rows:
        row['model_name'] = model_name
    sql = """
        INSERT IGNORE INTO qubit_rankings
            (model_name, backend, qubit, snapshot_date,
             viability_score, viability_rank, label,
             T1_us, T2_us, readout_error)
        VALUES
            (%(model_name)s, %(backend)s, %(qubit)s, %(snapshot_date)s,
             %(viability_score)s, %(viability_rank)s, %(label)s,
             %(T1_us)s, %(T2_us)s, %(readout_error)s)
    """
    conn = get_connection()
    cur  = conn.cursor()
    cur.executemany(sql, rows)
    inserted = cur.rowcount
    conn.commit()
    cur.close()
    conn.close()
    return inserted


# ── Read operations ───────────────────────────────────────────────────────────
def get_history_for_backend(backend: str) -> pd.DataFrame:
    """Load all calibration history for a backend — used for feature engineering."""
    sql = """
        SELECT backend, qubit, snapshot_date, T1_us, T2_us, readout_error
        FROM calibration_history
        WHERE backend = %s
        ORDER BY qubit, snapshot_date
    """
    conn = get_connection()
    df   = pd.read_sql(sql, conn, params=(backend,))
    conn.close()
    return df


def get_latest_rankings(backend: str = None,
                        model_name: str = 'model_a',
                        top_n: int = None) -> pd.DataFrame:
    """
    Get most recent rankings for a specific model.
    model_name: 'model_a' | 'model_b' | 'model_c' | 'all'
    """
    if model_name == 'all':
        model_filter = ""
        params = []
    else:
        model_filter = "AND model_name = %s"
        params = [model_name]

    date_sub = f"(SELECT MAX(snapshot_date) FROM qubit_rankings WHERE model_name = '{model_name}')" \
               if model_name != 'all' else \
               "(SELECT MAX(snapshot_date) FROM qubit_rankings)"

    backend_filter = ""
    if backend:
        backend_filter = "AND backend = %s"
        params.append(backend)

    limit = f"LIMIT {top_n}" if top_n else ""

    sql = f"""
        SELECT model_name, backend, qubit, viability_rank, viability_score,
               label, T1_us, T2_us, readout_error, snapshot_date
        FROM qubit_rankings
        WHERE snapshot_date = {date_sub}
        {model_filter}
        {backend_filter}
        ORDER BY model_name, backend, viability_rank
        {limit}
    """
    conn = get_connection()
    df   = pd.read_sql(sql, conn, params=params if params else None)
    conn.close()
    return df


def get_all_models_latest(backend: str = None) -> pd.DataFrame:
    """
    Get latest rankings for ALL three models side by side.
    Returns a merged DataFrame for comparison.
    """
    frames = []
    for m in ['model_a', 'model_b', 'model_c']:
        df = get_latest_rankings(backend=backend, model_name=m)
        if not df.empty:
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def get_ranking_history(backend: str, qubit: int,
                        model_name: str = 'model_a') -> pd.DataFrame:
    """Load ranking history for a specific qubit and model."""
    sql = """
        SELECT snapshot_date, model_name, viability_score, viability_rank,
               label, T1_us, T2_us, readout_error
        FROM qubit_rankings
        WHERE backend = %s AND qubit = %s AND model_name = %s
        ORDER BY snapshot_date
    """
    conn = get_connection()
    df   = pd.read_sql(sql, conn, params=(backend, int(qubit), model_name))
    conn.close()
    return df


def get_all_models_history(backend: str, qubit: int) -> pd.DataFrame:
    """Load ranking history for a qubit across ALL models."""
    sql = """
        SELECT snapshot_date, model_name, viability_score, label
        FROM qubit_rankings
        WHERE backend = %s AND qubit = %s
        ORDER BY snapshot_date, model_name
    """
    conn = get_connection()
    df   = pd.read_sql(sql, conn, params=(backend, int(qubit)))
    conn.close()
    return df


def get_all_dates(model_name: str = 'model_a') -> list:
    sql  = """
        SELECT DISTINCT snapshot_date
        FROM qubit_rankings
        WHERE model_name = %s
        ORDER BY snapshot_date
    """
    conn = get_connection()
    cur  = conn.cursor()
    cur.execute(sql, (model_name,))
    dates = [r[0] for r in cur.fetchall()]
    cur.close()
    conn.close()
    return dates


def get_summary_stats() -> dict:
    """Summary for dashboard header — covers all models."""
    conn = get_connection()
    cur  = conn.cursor(dictionary=True)

    # Days collected (from calibration, model-agnostic)
    cur.execute("SELECT COUNT(DISTINCT snapshot_date) as days FROM calibration_history")
    days = cur.fetchone()['days']

    # Latest date
    cur.execute("SELECT MAX(snapshot_date) as d FROM calibration_history")
    row    = cur.fetchone()
    latest = row['d'] if row else None

    # Active qubits
    cur.execute("SELECT COUNT(DISTINCT CONCAT(backend,'-',qubit)) as n FROM calibration_history WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM calibration_history)")
    qubits = cur.fetchone()['n']

    # Which models have data
    cur.execute("SELECT DISTINCT model_name FROM qubit_rankings ORDER BY model_name")
    models_live = [r['model_name'] for r in cur.fetchall()]

    cur.close()
    conn.close()
    return {
        'days_collected':  days,
        'qubits_latest':   qubits,
        'latest_date':     latest,
        'models_live':     models_live,
    }


def get_model_comparison_latest(backend: str = None) -> pd.DataFrame:
    """
    Returns a pivot comparing all three models' viability scores
    for each (backend, qubit) on the most recent date.
    Used by the Model Comparison tab in the dashboard.
    """
    all_df = get_all_models_latest(backend=backend)
    if all_df.empty:
        return pd.DataFrame()

    pivot = all_df.pivot_table(
        index=['backend', 'qubit'],
        columns='model_name',
        values='viability_score',
        aggfunc='mean'
    ).reset_index()

    # Also pull latest T1/T2/RE from model_a (they are hardware values, same for all)
    ref = all_df[all_df['model_name'] == 'model_a'][
        ['backend', 'qubit', 'T1_us', 'T2_us', 'readout_error', 'label']
    ].copy()

    pivot = pivot.merge(ref, on=['backend', 'qubit'], how='left')
    return pivot


def get_consistency_all_models(backend: str = None) -> pd.DataFrame:
    """7-day consistency across all models for the dashboard."""
    backend_filter = "AND backend = %s" if backend else ""
    params = (backend,) if backend else ()

    sql = f"""
        SELECT model_name, backend, qubit,
               COUNT(DISTINCT snapshot_date)       AS days_observed,
               SUM(label)                          AS days_viable,
               ROUND(AVG(viability_score), 4)      AS mean_score,
               ROUND(STDDEV(viability_score), 4)   AS score_std
        FROM qubit_rankings
        WHERE 1=1 {backend_filter}
        GROUP BY model_name, backend, qubit
        ORDER BY model_name, backend, mean_score DESC
    """
    conn = get_connection()
    df   = pd.read_sql(sql, conn, params=params if params else None)
    conn.close()
    return df